# Lab 12 — Imbalance & the accuracy illusion — the metric that cannot be fooled

> **Type-2 lab — student version.** Fill in each `# TODO` (the answer cells raise `NotImplementedError` until you do), answer the **Checkpoint** questions, and complete the reflection cell, then re-run top to bottom. The instructor solution lives in `labs/solutions/` and is not included here.

**Covers.** Chapter 12 — §12.5 (class imbalance and honest metrics), §12.4 (the failure-case clinic, Failure 5).

**Biomedical question.** My sleep scorer is 79% accurate. Is it any good — and would I notice if it had never once detected N1?
**Task type (§1.8).** Classification — choosing an evaluation metric that matches the claim under class imbalance.
**Information that must be preserved.** the **rare stages**. N1 is roughly 5% of a night; a headline metric that lets a scorer ignore N1 entirely hides exactly the epochs a clinician needs to see.
**Main assumptions.** the labels follow realistic AASM sleep prevalence (N2 dominates, N1 is rare); the *split* is already subject-independent — that battle is fought in **lab11**, so here `GroupKFold` is a given and the **metric** is the only variable.
**Primary diagnostic.** a **metric panel** — accuracy · balanced accuracy · macro-F1 · Cohen's kappa — read next to the **confusion matrix** and **per-class recall**, never a single headline number.
**Transfer challenge.** redo the argument for a *binary* detector at 3% prevalence (apnea vs none) and choose the operating point from a clinical constraint rather than from accuracy.

*Self-contained: one seeded synthetic multi-subject cohort of per-epoch feature vectors, `numpy` + `scikit-learn` only, no `bsp`, no data files, no network. Runs offline in well under a minute. Theme (§1.8): there is no single best metric — but there are **wrong** ones, and reporting accuracy on a 45%-N2 night is one of them.*

### Companion lab · *Biomedical Signal Processing & Data Analytics* (CM2013)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/labs/lab12_leakage_clinic_honest_validation/lab12_leakage_clinic_honest_validation.ipynb) [![View](https://img.shields.io/badge/view-static-orange)](https://farhad-abtahi.github.io/CM2013/nb/lab12_leakage_clinic_honest_validation.html) [![JupyterLite](https://img.shields.io/badge/run-JupyterLite-blue)](https://farhad-abtahi.github.io/CM2013/lite/lab/index.html?path=lab12_leakage_clinic_honest_validation.ipynb)

In [ ]:
# --- shared setup (reproducible; fully offline synthetic cohort) ---
import numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_predict, GroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             cohen_kappa_score, confusion_matrix, recall_score)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

STAGES = ["W", "N1", "N2", "N3", "REM"]
# Roughly a real night's architecture (Ch. 12 §12.5): N2 dominates, N1 is rare.
PREV   = np.array([0.10, 0.05, 0.45, 0.15, 0.25])
FEATS  = ["delta_bp", "sigma_bp", "alpha_bp", "emg_rms", "eog_var"]

# Class means in that 5-feature space. N1 is placed deliberately BETWEEN W and N2: it is the
# stage human scorers agree on least, so it must be genuinely hard here too -- rare AND ambiguous.
M = np.array([
    [-0.6, -0.3,  1.4,  1.2,  0.5],   # W   : alpha + muscle tone
    [-0.1, -0.1,  0.5,  0.7,  0.3],   # N1  : sits between W and N2 (ambiguous by design)
    [ 0.2,  1.3, -0.3,  0.2, -0.4],   # N2  : spindles -> sigma power
    [ 1.9,  0.1, -0.6,  0.1, -0.5],   # N3  : delta-heavy, easy
    [-0.4, -0.4,  0.1, -0.9,  1.6],   # REM : atonia + eye movement
])

def make_cohort(n_subj=12, per=180, prev=PREV, noise=0.85, offset_std=0.35, seed=2013):
    """Per-epoch FEATURE vectors for a multi-subject cohort with realistic stage prevalence.

    Each subject gets a small nuisance offset, so subjects are still held whole across the
    split -- but the offset is deliberately SMALL (0.35 vs class gaps ~1-2). Leakage is
    lab11's subject; this lab keeps the split honest and varies only the metric.
    """
    r = np.random.default_rng(seed)
    X, y, g = [], [], []
    for s in range(n_subj):
        off = r.normal(0, offset_std, size=len(FEATS))
        for c in r.choice(len(STAGES), size=per, p=prev):
            X.append(M[c] + off + r.normal(0, noise, size=len(FEATS)))
            y.append(int(c)); g.append(s)
    return np.array(X), np.array(y), np.array(g)

CV = GroupKFold(5)          # subject-independent throughout: whole subjects held out

## 1. Look at the prevalence *before* you look at a score
The single most useful number to know before reading any accuracy is **how often the majority class occurs** — because that is what a model that has learned nothing can score. Build the cohort and read its class balance.

In [ ]:
X, y, groups = make_cohort()
counts = np.bincount(y, minlength=len(STAGES))
majority_share = counts.max() / counts.sum()
print(f"X {X.shape}   subjects {len(np.unique(groups))}   features {FEATS}\n")
print(f"{'stage':>5} {'epochs':>7} {'share':>8}")
for k, s in enumerate(STAGES):
    print(f"{s:>5} {counts[k]:>7d} {counts[k]/counts.sum():>7.1%}")
print(f"\nmajority class = {STAGES[int(counts.argmax())]} at {majority_share:.1%}"
      f"  <-- the accuracy a 'always say the majority' rule already gets, for free")

plt.figure(figsize=(5.2, 2.8))
plt.bar(STAGES, counts / counts.sum(), color=["tab:gray"]*len(STAGES))
plt.axhline(majority_share, color="tab:red", ls="--", lw=1,
            label=f"free accuracy = {majority_share:.2f}")
plt.ylabel("share of epochs"); plt.title("A night is not balanced (Ch. 12 §12.5)")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()
# Checkpoint 1: before running any model — what accuracy would you have to BEAT before an
# accuracy number tells you anything at all? And which stage would you never notice missing?

## 2. The metric panel — four numbers, not one
Chapter 12 §12.5 asks for a *panel*, because each metric rewards something different:

| Metric | Rewards the majority guess? | Chance-corrected? |
|---|---|---|
| **accuracy** — fraction correct | **yes — dangerous** | no |
| **balanced accuracy** — mean per-class recall | no | no |
| **macro-F1** — unweighted mean of per-class F1 | no — punishes ignoring rare classes | no |
| **Cohen's kappa** — agreement above chance | no — 0 for *any* chance-level guesser | yes |

Build the panel once, then apply it unchanged to every model, starting with the "learned nothing" baseline.

In [ ]:
# TODO write metric_panel(y_true, y_pred) returning a dict with EXACTLY these four keys:
#     "accuracy", "balanced_acc", "macro_F1", "kappa"
#   using accuracy_score, balanced_accuracy_score, f1_score(..., average="macro") and
#   cohen_kappa_score. Then score the MAJORITY-CLASS baseline: get subject-independent
#   predictions for DummyClassifier(strategy="most_frequent") with cross_val_predict(...,
#   cv=CV, groups=groups), store them in yp_base, and store metric_panel(y, yp_base) in p_base.
#   All three names -- metric_panel, yp_base, p_base -- are used by the cells below and by the
#   sanity check, so bind them exactly.
raise NotImplementedError("TODO: implement this — see the comment above")
print("MAJORITY-CLASS BASELINE (predicts N2 for every single epoch):")
for k, v in p_base.items():
    print(f"    {k:14s} = {v:.3f}")
print(f"\n  it predicts exactly {len(set(yp_base.tolist()))} distinct label(s) and has learned NOTHING,")
print(f"  yet its accuracy is {p_base['accuracy']:.2f}. Its kappa is {p_base['kappa']:.2f}.")
# Checkpoint 2: accuracy says "half right", kappa says "zero skill". Which one is lying, and
# what exactly does kappa subtract that accuracy does not?

## 3. A real classifier — and the stage the headline hides
Now train something that genuinely learns. Score it the *same* subject-independent way, then
refuse to stop at the headline: print the **confusion matrix** and **per-class recall**, which is
where §12.5 says the story actually lives.

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=0)

# TODO get subject-independent predictions for `rf` with cross_val_predict(..., cv=CV,
#   groups=groups) into yp_plain, and its panel into p_plain. Also compute the per-class recall
#   vector into rec_plain with recall_score(y, yp_plain, average=None). All three names are used
#   below and by the sanity check, so bind them exactly.
raise NotImplementedError("TODO: implement this — see the comment above")
print("PLAIN RANDOM FOREST (subject-independent GroupKFold):")
for k, v in p_plain.items():
    print(f"    {k:14s} = {v:.3f}   (baseline {p_base[k]:.3f})")

cm_plain = confusion_matrix(y, yp_plain)
print("\nconfusion matrix — rows = TRUE stage, columns = PREDICTED")
print(f"{'':>5} " + " ".join(f"{s:>5}" for s in STAGES))
for k, s in enumerate(STAGES):
    print(f"{s:>5} " + " ".join(f"{v:>5d}" for v in cm_plain[k]))
print("\nper-class recall (of the true epochs of this stage, how many did we catch?)")
for k, s in enumerate(STAGES):
    flag = "   <-- essentially invisible" if rec_plain[k] < 0.15 else ""
    print(f"    {s:>4}: {rec_plain[k]:.3f}{flag}")
print(f"\n  headline accuracy {p_plain['accuracy']:.2f} looks strong — and it survives the fact that"
      f"\n  N1 recall is {rec_plain[1]:.2f}, because N1 is only {counts[1]/counts.sum():.0%} of the epochs.")
# Checkpoint 3: how much would the ACCURACY change if the model caught every single N1 epoch?
# Compute it in your head from the confusion matrix, then say why that answers the lab's question.

## 4. Make it visible — move the decision, not the metric
The forest is not broken; its **decision rule** is. `argmax` over the predicted probabilities
maximises expected accuracy, and under this prevalence the accuracy-optimal move is to almost never
say "N1". Divide each class probability by its **prior** before taking the argmax and you ask a
different question — *which stage is this epoch most surprising evidence for* — trading majority
accuracy for rare-class recall.

Predict what happens to each of the four metrics **before** you run it.

In [ ]:
prior = counts / counts.sum()

# TODO get the out-of-fold PROBABILITIES for `rf` (same CV, same groups) with
#   cross_val_predict(..., method="predict_proba") into P_oof. Then form prior-corrected
#   predictions yp_corr = (P_oof / prior).argmax(1), its panel p_corr, and its per-class
#   recall rec_corr. Bind exactly those four names.
raise NotImplementedError("TODO: implement this — see the comment above")
print("PRIOR-CORRECTED RANDOM FOREST (same model, same folds, different decision rule):")
for k, v in p_corr.items():
    arrow = "UP  " if v > p_plain[k] else ("DOWN" if v < p_plain[k] else "same")
    print(f"    {k:14s} = {v:.3f}   ({arrow} from {p_plain[k]:.3f})")
print("\nper-class recall, plain -> prior-corrected")
for k, s in enumerate(STAGES):
    print(f"    {s:>4}: {rec_plain[k]:.3f} -> {rec_corr[k]:.3f}")
print(f"\n  N1 recall went {rec_plain[1]:.3f} -> {rec_corr[1]:.3f} "
      f"({rec_corr[1]/max(rec_plain[1], 1e-9):.1f}x) while ACCURACY FELL "
      f"{p_plain['accuracy']:.3f} -> {p_corr['accuracy']:.3f}.")
# Checkpoint 4: accuracy went down and kappa went down, yet balanced accuracy went UP and the
# model now actually detects the rare stage. Which model would you ship, and — crucially — what
# would you have to state about the CLAIM to make that choice defensible?

## 5. Read the panel side by side — and watch the ranking flip
One table, three models, four metrics. The point of §12.5 is not that one metric is "right": it is
that **the ranking depends on the metric**, so the metric has to be chosen from the clinical
question *before* the numbers arrive — otherwise you will pick the metric that flatters the model
you already built.

In [ ]:
names = ["majority baseline", "plain forest", "prior-corrected forest"]
panels = [p_base, p_plain, p_corr]
keys = ["accuracy", "balanced_acc", "macro_F1", "kappa"]

print(f"{'model':>24} " + " ".join(f"{k:>13}" for k in keys))
for nm, p in zip(names, panels):
    print(f"{nm:>24} " + " ".join(f"{p[k]:>13.3f}" for k in keys))

print("\nranking by each metric (best first):")
for k in keys:
    order = sorted(range(3), key=lambda i: -panels[i][k])
    print(f"    {k:14s}: " + "  >  ".join(names[i] for i in order))

fig, ax = plt.subplots(figsize=(7.4, 3.4))
w, xs = 0.26, np.arange(len(keys))
for i, (nm, p) in enumerate(zip(names, panels)):
    ax.bar(xs + (i - 1) * w, [p[k] for k in keys], width=w, label=nm)
ax.axhline(majority_share, color="tab:red", ls="--", lw=1,
           label=f"majority share ({majority_share:.2f})")
ax.set_xticks(xs); ax.set_xticklabels(keys); ax.set_ylabel("score"); ax.set_ylim(0, 1)
ax.set_title("The same three models, four metrics — the ranking is not the same")
ax.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()
print("Read the bars for the baseline: high on accuracy, ZERO on kappa. That single contrast is")
print("Failure 5 of the failure-case clinic (§12.4) — imbalance hidden by accuracy — made visible.")

### Live sanity check
A metric claim you never verify is untrustworthy. These asserts run on the numbers computed above
and encode the lab's three claims: a model that has learned nothing still scores ~half on accuracy
while kappa correctly reports zero; the accurate-looking forest essentially never detects the rare
stage; and moving the decision rule flips the metric ranking — accuracy down, rare-class recall up.

In [ ]:
# --- live sanity check: every number below was computed by the cells above ---
# (1) the "learned nothing" baseline: high accuracy, ZERO skill.
assert len(set(yp_base.tolist())) == 1, "the majority baseline must predict a single label"
assert p_base["accuracy"] > 0.40, f"baseline accuracy should be ~the majority share, got {p_base['accuracy']:.3f}"
assert abs(p_base["kappa"]) < 0.02, f"kappa must be ~0 for a chance-level guesser, got {p_base['kappa']:.3f}"
assert p_base["macro_F1"] < 0.20 and p_base["balanced_acc"] < 0.25, \
    "macro-F1 and balanced accuracy must also expose the baseline"

# (2) a genuinely skilful model can still be blind to the rare stage — and accuracy will not say so.
assert p_plain["kappa"] > 0.5, "the forest really has learned something (kappa well above chance)"
assert p_plain["accuracy"] > p_base["accuracy"] + 0.20, "and it beats the baseline on accuracy"
assert rec_plain[1] < 0.15, f"N1 recall should be near zero, got {rec_plain[1]:.3f}"
assert rec_plain[2] > 0.80, "...while the majority stage N2 is caught almost every time"

# (3) the ranking FLIPS with the metric: accuracy and kappa fall, balanced accuracy and N1 recall rise.
assert p_corr["accuracy"] < p_plain["accuracy"], "prior correction must cost accuracy"
assert p_corr["balanced_acc"] > p_plain["balanced_acc"], "...and must buy balanced accuracy"
assert rec_corr[1] > 3.0 * rec_plain[1], f"N1 recall must improve sharply, {rec_plain[1]:.3f} -> {rec_corr[1]:.3f}"
best_acc = max(range(3), key=lambda i: panels[i]["accuracy"])
best_bal = max(range(3), key=lambda i: panels[i]["balanced_acc"])
assert best_acc != best_bal, "the whole lesson: accuracy and balanced accuracy must not crown the same model"

print("sanity check PASSED:")
print(f"  baseline      accuracy {p_base['accuracy']:.2f}  kappa {p_base['kappa']:.2f}  (learned nothing)")
print(f"  plain forest  accuracy {p_plain['accuracy']:.2f}  kappa {p_plain['kappa']:.2f}"
      f"  N1 recall {rec_plain[1]:.2f}")
print(f"  prior-corr.   accuracy {p_corr['accuracy']:.2f}  kappa {p_corr['kappa']:.2f}"
      f"  N1 recall {rec_corr[1]:.2f}")
print(f"  best by accuracy = '{names[best_acc]}'  |  best by balanced accuracy = '{names[best_bal]}'")

## Reflection

This reflection is for your own practice — there is nothing to submit. What matters is the *reasoning*, not hitting a particular number.

1. **Stable vs changed.** Which verdict held across *every* metric in the panel, and which reversed the moment you switched metric? Name the pair of models and the pair of metrics that disagree.
2. **The number you must quote first.** Why is the majority-class share the number that has to appear *before* any accuracy in a report — and what does an accuracy below it tell you?
3. **Kappa is not immune.** Kappa is chance-corrected, yet here it *fell* for the model that detects the rare stage better. Explain that using its definition, and say why §12.5 insists kappa be read *alongside* per-class recall and the confusion matrix rather than alone.
4. **Match the metric to the claim.** State the clinical claim under which you would ship the plain forest, and the claim under which you would ship the prior-corrected one. Each claim should make its metric obvious.
5. **New cohort.** A paediatric cohort has a different stage balance. Which of the four metrics would you expect to move *purely because the prevalence changed*, with the model untouched — and what would you report to make that visible?

**Rule out.** Reporting **accuracy as the headline on imbalanced classes** is ruled out — not a matter of taste. You measured it: a rule that predicts "N2" for every epoch and has learned nothing at all scores an accuracy near the majority share while its Cohen's kappa is exactly 0.00, its balanced accuracy is 1/5, and its macro-F1 collapses. The same illusion survives into a model that *has* learned: the forest reached a strong-looking accuracy while catching under 15% of N1 epochs, because N1 is too rare to move the average. That breaks the **§1.8** requirement that the rare stages survive the evaluation, so the reported number no longer supports the clinical claim it is quoted for. This is Failure 5 of the failure-case clinic (§12.4). The disciplined report leads with the **confusion matrix** and a **panel** — per-class recall, balanced accuracy, macro-F1, kappa — with the majority-class share quoted alongside, and it names the metric *before* the model is chosen. Two further wrong moves to rule out: treating kappa as *immune* to imbalance (it is chance-corrected, not prevalence-free — you watched it fall for the better rare-class detector), and "fixing" a low macro-F1 by changing the metric instead of the decision rule.

> *Your answers here.*

**Where the other half of Chapter 12 lives.** Splitting to the claim and the leakage failures (§12.3, §12.4 Failures 1–4) are exercised end-to-end in **lab11**, which builds the inflated-vs-honest gap; this lab holds the split fixed and subject-independent on purpose so that the metric is the only thing that moves. Calibration (§12.7) and external validation (§12.8) are carried by the Ch12 companion notebook and the capstone.

---
*Type-2 lab for **Biomedical Signal Processing & Data Analytics**. Synthetic cohort; illustrative numbers. The lesson is the discipline, not the exact score: choose the metric from the claim, then report the panel — never a lone headline.*